# Setup notebook — one-time bootstrap

Run this notebook *once* on a new machine. It installs the Python deps,
pulls the two Docker images the pipeline needs, and verifies the bundled
FreeCAD / ParaView. After this you should be able to open
`workflow_explorer.ipynb` and click your way through a real run.

**What it does NOT install:**
* ANSYS host install + license server — you must already have one of
  ETH (`1801@lic-ansys-research.ethz.ch`), PSI (`1055@winlic03.psi.ch`),
  CERN, or your own.
* Docker Desktop — install separately from docker.com.
* FreeCAD / ParaView — these are *bundled* under `tools/` in the repo;
  this notebook only verifies them.

The cells use `!` shell magic so they run in the same kernel you'll use for
the main notebook. If you'd rather run from PowerShell, every cell here has
a one-line CLI equivalent in its description.

## 1. Python version + pip dependencies

Requires Python 3.12 (pinned by `pyproject.toml`). Equivalent CLI:
```
pip install -e .[notebook]
```

In [ ]:
import sys
print('Python:', sys.version)
ok = sys.version_info[:2] == (3, 12)
print('Required: 3.12 (matches pyproject.toml `requires-python = ">=3.12,<3.13"`)')
print('Status:', 'OK' if ok else 'WRONG VERSION - install Python 3.12 before continuing')

In [ ]:
# Editable install picks up local changes to scripts/main/* without reinstall.
# The `notebook` extra pulls jupyterlab + ipywidgets.
from pathlib import Path
import sys, subprocess
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
print('Installing from:', REPO_ROOT)
rc = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', f'{REPO_ROOT}[notebook]'],
    capture_output=False)
print('pip exit code:', rc.returncode)

## 2. Verify Python imports

In [ ]:
from pathlib import Path
import sys
NB = Path.cwd() if Path.cwd().name == 'notebooks' else (Path.cwd() / 'notebooks')
if str(NB) not in sys.path:
    sys.path.insert(0, str(NB))
import workflow_viz as v
v.display_environment_check(include_docker_images=False)

## 3. Docker registry

Pick the registry your ANSYS Docker images live in. The defaults work for
PSI; CERN users change to `registry.cern.ch/chart-magnum`. The value is
saved as the `REGISTRY_PREFIX` env var (picked up by the docker-compose
files and `mesh_to_lsdyna.py`).

In [ ]:
import os, ipywidgets as W
from IPython.display import display
registry_dd = W.Combobox(
    value='gitea.psi.ch/vanden_j',
    options=['gitea.psi.ch/vanden_j', 'registry.cern.ch/chart-magnum'],
    description='REGISTRY_PREFIX:',
    layout=W.Layout(width='520px'),
    style={'description_width': '160px'},
    ensure_option=False)
display(registry_dd)

## 4. Pull Docker images

Equivalent CLI (single line per image):
```
docker pull <REGISTRY_PREFIX>/mechanical:25.2
docker pull <REGISTRY_PREFIX>/lsdyna:25.2
```
These are large (~5 GB each). Skip this cell if you already pulled them.

In [ ]:
import subprocess, os
prefix = registry_dd.value.strip()
os.environ['REGISTRY_PREFIX'] = prefix
for img in (f'{prefix}/mechanical:25.2', f'{prefix}/lsdyna:25.2'):
    print(f'\n=== docker pull {img} ===')
    rc = subprocess.run(['docker', 'pull', img])
    print(f'exit code: {rc.returncode}')

## 5. Verify Docker daemon + images + bundled tools

In [ ]:
summary = v.display_environment_check(
    include_docker_images=True,
    registry_prefix=registry_dd.value.strip())
print()
print('Section summaries:', summary)
if all(summary.values()):
    print('All checks passed - open workflow_explorer.ipynb next.')
else:
    print('Some checks failed - see the tables above.')

## Notes

* **ANSYS license server.** `WorkflowRunner` auto-detects ETH / PSI / CERN
  via TCP probe; override with the `ANSYS_LICENSE_SERVER` env var if your
  cluster has a different server.
* **HPC backend.** `workflow_explorer.ipynb` exposes a generic HPC widget
  (any cluster reachable by SSH). Cluster-neutral env vars
  `HPC_{USER,HOST,REMOTE_BASE}` control the underlying transport, with
  defaults pointing at ETH Euler.
* **Re-running this notebook is safe.** `pip install -e` is idempotent;
  `docker pull` is a no-op if the image is already up-to-date.